In [3]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
from IPython.display import display, Markdown
import os
from pathlib import Path
from agents.extensions.models.litellm_model import LitellmModel

load_dotenv(override=True)
print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

OpenRouter API Key found: True


In [4]:
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')


In [2]:
import json
json.dumps({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})

'{"include_images": false, "include_answer": true, "search_depth": "advanced", "max_results": 10}'

In [ ]:
import json

params = {
      "command": "npx",
      "args": [
        "-y",
        "mcp-remote",
        f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}"
      ],
      "env": {
        "DEFAULT_PARAMETERS": json.dumps({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})
      }
}

tool_filter=create_static_tool_filter(blocked_tool_names=["tavily_research","tavily_map","tavily_crawl","tavily_skill"])

In [6]:
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server_python = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "ghcr.io/hrrodan/agent-workspace-mcp:latest" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

In [10]:
abs_tmp_dir

'/home/martin/Python/Projects/Github/agents/6_mcp/tmp'

In [14]:
instructions = "You have tools available, use them if necessary."
request = "Check the docs from functools online and write a small example script in your workspace. Use the parameters include_answer: True, search_depth: advanced"
model = "openrouter/google/gemini-3-flash-preview"

In [15]:

async with MCPServerStdio(params=params,tool_filter=tool_filter ,client_session_timeout_seconds=30) as mcp_server:
    async with mcp_server_python as mcp_server_python:
        agent = Agent(name="tool_manager", instructions=instructions, model=LitellmModel(model), mcp_servers=[mcp_server, mcp_server_python])
        with trace("tool_manager"):
            result = await Runner.run(agent, request, max_turns=30)
        display(Markdown(result.final_output))



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



I have consulted the `functools` documentation and created a comprehensive example script in the workspace.

### Key `functools` Features Included:
1.  **`@lru_cache`**: Automatically caches function results based on arguments, turning an $O(2^n)$ recursive Fibonacci function into $O(n)$.
2.  **`partial`**: Pre-fills function arguments to create new, specialized functions (e.g., creating `square` and `cube` from a general `power` function).
3.  **`@singledispatch`**: Enables functional "overloading" at runtime based on the type of the first argument, allowing clean, type-specific logic.
4.  **`@wraps`**: A decorator used inside custom decorators to ensure the original function's metadata (like `__name__` and `__doc__`) is preserved.

### Example Script Output:
```text
--- 1. LRU Cache (Fibonacci) ---
Fibonacci(35) = 9227465 (Time: 0.000070s)
Cache Performance: CacheInfo(hits=33, misses=36, maxsize=None, currsize=36)

--- 2. Partial Functions ---
Square of 5: 25
Cube of 5: 125

--- 3. Single Dispatch ---
Integer: 42
List of length 3: 1, 2, 3
Generic: Hello

--- 4. Wraps Decorator ---
Calling say_hello...
Hello, World!
Function Name: say_hello
Function Docstring: Greets the user.
```

The script `functools_example.py` has been saved and executed successfully in the workspace.